# 03 – Python and SQL Together

Now you combine both: run SQL from Python, work with the results and draw a chart.

> **SOLUTION** – try the exercises yourself first!

In [ ]:
# Run this cell first! It loads the helper functions.
import os
if os.path.isdir("Exercises"):          # started in the project folder
    os.chdir("Exercises")
elif not os.path.exists("helpers.py"):  # started in the solutions folder
    os.chdir("..")
from helpers import *
import matplotlib.pyplot as plt

setup_database()

## Python and SQL together

The result of `sql()` is a **pandas DataFrame** – a table in Python.
Some useful things you can do with a DataFrame `df`:

```python
len(df)                 # number of rows
df["price"]             # one column
df["price"].mean()      # average of a column
df[df["price"] > 50]    # filter rows
df.head(3)              # first 3 rows
df["price"].tolist()    # column as a Python list
```

### Exercise 1 – Query result in a variable

Save all products (`SELECT * FROM products`) in the variable `df`.
Then save the number of rows in `number_of_products`.

In [ ]:
df = sql("SELECT * FROM products")
number_of_products = len(df)

print(number_of_products)
df.head()

In [ ]:
# Check your answer
check(number_of_products == 77, "use len(df)")

### Exercise 2 – Calculate in Python

Using `df` from exercise 1, calculate `average_price` with pandas (`df["price"].mean()`) and round it to 2 decimals.

Compare: in SQL this is `SELECT ROUND(AVG(price)::numeric, 2) FROM products`.

In [ ]:
average_price = round(df["price"].mean(), 2)
print(average_price)

In [ ]:
# Check your answer
check(average_price == 28.87, "round(df[\"price\"].mean(), 2)")

### Exercise 3 – Loop over query results

Run a query that returns `productname` and `price` of products with price over 50.
Then use a **for loop** over the rows to build the list `lines` with texts like `"Côte de Blaye costs 263.5"`.

Tip: `for index, row in result.iterrows():` gives you each row; `row["productname"]` gives a value.

In [ ]:
result = sql("""
SELECT productname, price
FROM products
WHERE price > 50
""")
lines = []

for index, row in result.iterrows():
    lines.append(f"{row['productname']} costs {row['price']}")

for line in lines:
    print(line)

In [ ]:
# Check your answer
check(len(lines) == 7 and "Côte de Blaye costs 263.5" in lines,
      "7 products cost more than 50")

### Exercise 4 – A function with a query parameter

Write a function `products_of_country(country)` that returns a DataFrame with the `productname` of all products whose **supplier** is in that country.

Never glue user input into SQL with `+` or f-strings (SQL injection!). Use a **parameter** instead:

```python
sql("SELECT * FROM customers WHERE country = :c", {"c": "Spain"})
```

Tip: you need a JOIN between `products` and `suppliers`.

In [ ]:
def products_of_country(country):
    return sql("""
        SELECT p.productname
        FROM products p
        JOIN suppliers s ON p.supplierid = s.supplierid
        WHERE s.country = :country
    """, {"country": country})

products_of_country("Japan")

In [ ]:
# Check your answer
check(sorted(products_of_country("Japan")["productname"]) ==
      ["Genen Shouyu", "Ikura", "Konbu", "Longlife Tofu",
       "Mishi Kobe Niku", "Tofu"]
      and len(products_of_country("Sweden")) == 5,
      "JOIN suppliers s ON p.supplierid = s.supplierid WHERE s.country = :country")

### Exercise 5 – A chart from SQL data

1. Write a query `top_countries` with `country` and the number of customers as `customers`, **top 10** countries, most first.
2. Draw a bar chart with `plt.bar(x_values, y_values)`.

In [ ]:
top_countries = sql("""
SELECT country, COUNT(*) AS customers
FROM customers
GROUP BY country
ORDER BY customers DESC
LIMIT 10
""")

plt.bar(top_countries["country"], top_countries["customers"])

plt.title("Customers per country")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Check your answer
check(len(top_countries) == 10
      and top_countries["customers"].tolist()[:2] == [13, 11],
      "GROUP BY country, ORDER BY ... DESC, LIMIT 10")

### Exercise 6 – Save Python data in the database

You can also **write** data to the database. Create a DataFrame from a Python list of dictionaries and save it as table `my_favorites`:

```python
df = pd.DataFrame([{...}, {...}])
df.to_sql("table_name", engine, index=False, if_exists="replace")
```

Save at least **3** of your favorite products with the columns `productname` and `rating` (1–5). Then read the table back with `sql()` into `saved`.

In [ ]:
favorites = pd.DataFrame([
    {"productname": "Chai", "rating": 5},
    {"productname": "Ikura", "rating": 4},
    {"productname": "Tofu", "rating": 3},
])

favorites.to_sql("my_favorites", engine, index=False, if_exists="replace")

saved = sql("SELECT * FROM my_favorites")
saved

In [ ]:
# Check your answer
check(len(saved) >= 3 and list(saved.columns) == ["productname", "rating"],
      "use to_sql(\"my_favorites\", engine, index=False, if_exists=\"replace\")")

---
🎉 **Congratulations – you finished all beginner exercises!**

Next steps:
- `SQL/QUESTIONS.sql` – harder SQL questions
- `postgres_northwind_data.ipynb` – load data and query it yourself
- `postgres_apartment_data.ipynb` – a real dataset of apartments